## Python enables you to control multiple systems 

* Data pipelines are often written in Python, but almost all of the transformation logic is performed by a data processing engine (Spark in our case)

* The data processing engine (such as Spark, Snowflake, or BigQuery) operates as a separate system. Our Python code sends instructions to these engines, which perform the heavy data processing.

* Python's strength is being able to connect and work with almost any data system. Most data systems have a Python API.

#### Example

We use the `jdbc` mechanism to connect Spark to Postgres.

* Python is used to define the sequence of tasks (Extract -> Transform -> Load), while the data processing system executes the steps.

#### Exercise [5 min]

What is wrong with the code below?

In [ ]:
# Start a SparkSession
from pyspark.sql import Row, SparkSession

spark = (
    SparkSession.builder.appName("02_pipeline_design").master("local[*]").getOrCreate()
)

# Connection properties
url = "jdbc:postgresql://postgres:5432/ecommerce"
properties = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}

df = spark.read.jdbc(url=url, table="public.customer", properties=properties)

customer_json = df.collect()

In [ ]:


len(customer_json)

* we are pulling the entire data into Python process unnecessarily

## Create SCD2 tables with MERGE INTO

* We saw how SCD2 table tracks every change to an attribute with a new row
* Typically SCD2 tables have 3 main columns to represent state
  - `valid_from`: represents the start of the time range for this row to be considered active.
  - `valid_to`: represents the end of the time range for this row to be considered active
  - `is_current`: Flag to indicate the latest state of the data. With a `select * from scd2 where is_current=True` should give the same number of rows as the source table.
* Let's go over the scenarios we need to cover when building an SCD2 table
  - New customer:
    - [Insert] Add new row to dim_customer with valid_from as the created_at to valid_to set to some arbitrarily long future, and is_current = True
  - Update to an existing customer
    - [Update] Update existing (is_current = True) row for this customer and set its valid_to to curent updated_at and set its is_current = False
    - [Insert] Add new row to dim_customer with valid_from as the updated_at to valid_to set to some arbitrarily long future, and is_current = True
  - Delete existing customer:
    - [Update] Update existing (is_current = True) row for this customer and set valid_to to now() and its is_current = False. Note, we use now() since we cannot track deletes on a table (unless we use CDC)
* This operation is know as an `UPSERT` (update + insert)
* UPSERT used to be performed as complex multi steps, however with `MERGE INTO` we can do this as one SQL query

#### Example

Creating an SCD2 version of `dim_customer` as `dim_customer_scd2`.

In [ ]:
import argparse

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}

TABLE_NAME = "local.warehouse.dim_customer_scd2"


def extract(
    spark: SparkSession
) -> dict[str, DataFrame]:

    customer_df = spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.customer
        ) customer""",
        properties=JDBC_PROPERTIES,
    )
    customer_address_df = spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.customer_address
        ) customer_address""",
        properties=JDBC_PROPERTIES,
    )
    return {"customer": customer_df, "customer_address": customer_address_df}


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    input_dfs["customer"].createOrReplaceTempView("customer")
    input_dfs["customer_address"].createOrReplaceTempView("customer_address")

    return spark.sql("""
        SELECT
            c.customer_id,
            c.email,
            c.full_name,
            c.phone,
            c.status,
            c.created_at,
            c.updated_at,
            COLLECT_LIST(
                STRUCT(
                    ca.is_default,
                    CONCAT(ca.line1, ', ', ca.city, ', ', ca.state, ', ', ca.country) AS address
                )
            ) AS addresses
        FROM customer c
        LEFT JOIN customer_address ca USING (customer_id)
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    """)


def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        output_df.withColumn("valid_from", F.col("created_at")).withColumn(
            "valid_to", F.lit(None).cast("timestamp")
        ).withColumn("is_current", F.lit(True)).writeTo(TABLE_NAME).createOrReplace()
        return

    output_df.createOrReplaceTempView("staged")

    spark.sql(f"""
        WITH customers_with_updates AS (
            SELECT s.*
            FROM staged s
            JOIN {TABLE_NAME} t
                ON s.customer_id = t.customer_id  -- customer exists in dim_customer
            WHERE s.updated_at > t.updated_at     -- upstream change is newer than current dim record
            AND t.is_current = TRUE               -- only look at the most current version in dim_customer
        )
        MERGE INTO {TABLE_NAME} t  -- target dim_customer to update
        USING (
            SELECT customer_id AS join_key, *
            FROM staged  -- new customers to be INSERTED, existing customers to be EXPIRED

            UNION ALL

            SELECT NULL AS join_key, *
            FROM customers_with_updates  -- existing customers with updates, new version to be INSERTED
        ) s
        ON t.customer_id = s.join_key  -- natural key for customer

        WHEN MATCHED
            AND t.is_current = TRUE
            AND s.updated_at > t.updated_at  -- expire current row only if upstream has a newer update
        THEN UPDATE SET
            t.is_current = FALSE,
            t.valid_to   = s.updated_at

        WHEN NOT MATCHED  -- insert new customers and new versions of updated customers
        THEN INSERT (
            customer_id,
            email,
            full_name,
            phone,
            status,
            created_at,
            updated_at,
            addresses,
            valid_from,
            valid_to,
            is_current
        )
        VALUES (
            s.customer_id,
            s.email,
            s.full_name,
            s.phone,
            s.status,
            s.created_at,
            s.updated_at,
            s.addresses,
            s.updated_at,
            NULL,
            TRUE
        )

        WHEN NOT MATCHED BY SOURCE AND t.is_current = TRUE  -- customer no longer in source, soft delete
        THEN UPDATE SET
            t.is_current = FALSE,
            t.valid_to   = current_timestamp()
        """)


def run(spark: SparkSession) -> None:
    output_df = transform(extract(spark))
    load(output_df, spark)

* The UPSERT Logic is implemented in the `load` function.
* We do the following
  1. Identify all the rows to be updated and put them in a CTE `customers_with_updates.`
  2. Create a union of `customers_with_updates` and all the incoming data. This can result in duplicate customer records, as the customer row appears in both `customers_with_updates` and `staged`. This is necessary because we need both to update existing records and insert new ones.
  3. Use `WHEN NOT MATCHED THEN INSERT` to add new and updated customers.
  4. Use `WHEN MATCHED` to update existing customers.
  5. We use the `WHEN NOT MATCHED BY SOURCE` to identify rows in the dim_customer_scd2 table but not in the source data, meaning this row was deleted from the source.
* Let's look at how this will work

In [ ]:
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
spark.sql("drop table if exists local.warehouse.dim_customer_scd2")

In [ ]:
run(
    spark
)

In [ ]:
spark.table("local.warehouse.dim_customer_scd2").limit(3).toPandas()

In [ ]:
# selecting a customer_id to test
customer_id = (
    spark.table("local.warehouse.dim_customer_scd2")
    .filter(F.col("created_at") < "2025-04-01")
    .select(F.col("customer_id"))
    .collect()[0][0]
)

from pyspark.sql import functions as F

spark.table("local.warehouse.dim_customer_scd2").filter(
    F.col("customer_id") == customer_id
).limit(3).toPandas()

In [ ]:
%%sql
UPDATE customer
SET
    status     = 'suspended',
    updated_at = '2025-07-16 09:00:00+00:00'
WHERE customer_id = :customer_id;

UPDATE customer_address
SET
    line1      = '999 New Street',
    city       = 'Chicago',
    updated_at = '2025-07-16 09:00:00+00:00'
WHERE customer_id = :customer_id
  AND is_default  = True;

In [ ]:
run(spark)

In [ ]:
# ensuring change is reflected in the current spark session
spark.catalog.refreshTable("local.warehouse.dim_customer_scd2")

In [ ]:
from pyspark.sql import functions as F

spark.table("local.warehouse.dim_customer_scd2").filter(
    F.col("customer_id") == customer_id
).limit(3).toPandas()
# should show exactly 2 rows one current and the other one not

#### Exercise [10 min]

Create a `dim_product_scd2` based on the source `product` table. Here is the transformation function.

```python
def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    return input_dfs["product"].select(
        F.col("product_id"),
        F.col("name").alias("product_name"),
        F.col("brand"),
        F.col("base_price"),
        F.col("status"),
        F.col("updated_at"),
        F.col("created_at"),
    )
```


In [ ]:


from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

TABLE_NAME = "local.warehouse.dim_product_scd2"


def extract(
    spark: SparkSession
) -> dict[str, DataFrame]:
    product_df = spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.product
        ) customer""",
        properties=JDBC_PROPERTIES,
    )
    return {"product": product_df}


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    return input_dfs["product"].select(
        F.col("product_id"),
        F.col("name").alias("product_name"),
        F.col("brand"),
        F.col("base_price"),
        F.col("status"),
        F.col("updated_at"),
        F.col("created_at"),
    )


def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        output_df.withColumn("valid_from", F.col("created_at")).withColumn(
            "valid_to", F.lit(None).cast("timestamp")
        ).withColumn("is_current", F.lit(True)).writeTo(TABLE_NAME).createOrReplace()
        return

    output_df.createOrReplaceTempView("staged")

    spark.sql(f"""
        WITH products_with_updates AS (
            SELECT b.*
            FROM staged b
            JOIN {TABLE_NAME} d
                ON b.product_id = d.product_id
            WHERE b.updated_at > d.updated_at
            AND d.is_current = true
        )
        MERGE INTO {TABLE_NAME} t
        USING (
            SELECT product_id AS join_key, * FROM staged
            UNION ALL
            SELECT NULL AS join_key, * FROM products_with_updates
        ) s
        ON t.product_id = s.join_key

        WHEN MATCHED
            AND t.is_current = true
            AND s.updated_at > t.updated_at
        THEN UPDATE SET
            t.is_current = false,
            t.valid_to = s.updated_at

        WHEN NOT MATCHED
        THEN INSERT (
            product_id,
            product_name,
            brand,
            base_price,
            status,
            updated_at,
            created_at,
            valid_from,
            valid_to,
            is_current
        ) VALUES (
            s.product_id,
            s.product_name,
            s.brand,
            s.base_price,
            s.status,
            s.updated_at,
            s.created_at,
            s.updated_at,
            NULL,
            TRUE
        )

        WHEN NOT MATCHED BY SOURCE
            AND t.is_current = true
        THEN UPDATE SET
            t.is_current = false,
            t.valid_to = current_timestamp()
    """)


def run(spark: SparkSession) -> None:
    output_df = transform(extract(spark))
    load(output_df, spark)

**The following code should work as expected**

In [ ]:
spark.sql("drop table if exists local.warehouse.dim_product_scd2")

In [ ]:
run(
    spark
)

In [ ]:
spark.table("local.warehouse.dim_product_scd2").limit(3).toPandas()

In [ ]:
# selecting a customer_id to test
product_id = (
    spark.table("local.warehouse.dim_product_scd2")
    .filter(F.col("created_at") < "2025-04-01")
    .select(F.col("product_id"))
    .collect()[0][0]
)

from pyspark.sql import functions as F

spark.table("local.warehouse.dim_product_scd2").filter(
    F.col("product_id") == product_id
).limit(3).toPandas()

In [ ]:
%%sql
UPDATE product
SET
    name       = 'some_new_product',
    base_price = 100,
    updated_at = '2025-07-16 09:00:00+00:00'
WHERE product_id = :product_id;

In [ ]:
run(spark)

In [ ]:
# ensuring change is reflected in the current spark session
spark.catalog.refreshTable("local.warehouse.dim_product_scd2")

In [ ]:
from pyspark.sql import functions as F

spark.table("local.warehouse.dim_product_scd2").filter(
    F.col("product_id") == product_id
).limit(3).toPandas()
# should show exactly 2 rows one current and the other one not

## Backfills are inevitable, design your pipelines for them

* Every data pipeline will encounter issues over time. These issues could stem from bad input data or from a bug in the pipeline's code.

* In both cases, we will need to "replay" (meaning, re-execute) the data pipeline to correct mistakes in previous outputs.

* Backfill refers to re-running data pipelines for older data.

#### Example 

Now, consider an instance where a bug is found in the fct_order_lines pipeline. Suppose the issue has persisted for 10 months.

* The solution will be to fix the bug and re-run the pipeline for the failed months.
* This assumes we still have access to the source data for the past 10 months.

* However, this assumption may not always hold, especially in cases such as:
  - Extracting data from an external source
  - Extracting data from an application table that can be deleted or updated
  - Extracting data from a source that periodically purges old data

**Solution**:

* Keeping a copy of source data in our system ensures consistent historical access and enables pipeline re-runs when needed.
* This set of tables that are copies of source data used to be called raw/base/stage, etc.
In the medallion architecture (which we will discuss in the next section), this copy is referred to as the `bronze` layer, which simply means the first and rawest stage of data storage.
* Given the cheap data storage cost, most companies store a copy of their source data for easy access and re-runnability.
* With this approach, introduce new tables in your system that mirror the data from the upstream sources.

#### Exercise [5 min]

Assume that you have identified a bug in the `dim_product_scd` pipeline. You have identified that this bug has existed for the past 3 months. What are your next steps? How will you fix the data?

**Solution** 
* In this case you will loose the changelog for the past 3 months, since the upstream product table can be updated.
* You will need to
  - [Clean up] the rows for the past 3 months in dim_product_scd2 (you can try to recover data depending on the complexity of the issue)
  - [Re-run] the pipeline for the date range of the past 3 months
* We may loose some change data, but that is to be expected.


#### Example

Run this script to create the `bronze.product` table. Use it as the source in the `dim_product_scd2` pipeline.

In [ ]:
import argparse

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}
TABLE_NAME = "local.bronze.product"


def run(spark: SparkSession) -> None:
    spark.read.jdbc(
        url=JDBC_URL,
        table=f"""(
            SELECT *
            FROM public.product
        ) product""",
        properties=JDBC_PROPERTIES,
    ).writeTo(TABLE_NAME).createOrReplace()

In [ ]:
# "local.warehouse.dim_product_scd2" extract will be changed to use the bronze table


def extract(
    spark: SparkSession, start_time: str, end_time: str
) -> dict[str, DataFrame]:
    product_df = spark.table("local.bronze.product")
    return {"product": product_df}

* Backfills work without problems, only if our load functions are appropriate.
* Let’s go over the load function types we use for our facts and dimensions.
  * Fact: We use overwritePartition, which will ensure that when we re-run a pipeline, the appropriate partitions are overwritten, ensuring no duplicates or partial data
  * Snapshot dimension: We use createOrReplace, so re-running them will just create a new table
  * SCD2 dimension: We only update or insert because we identify only new rows using the `incoming _data.updated_at > existing_data.updated_at` and `existing_data.is_current=True`. This logic will ensure only new data gets inserted or updated. However, we will still need to manually clean up data for the time range during which the data/code was incorrect.

* When running backfills, there are scaling concerns. We can use one of the following approaches
| Run Type | Description | When To Use | Concerns |
|----------|-------------|-------------|----------|
| Serial | Run the pipeline one time range at a time, in order | When pipeline logic depends on previous runs (e.g., SCD2, where each run’s output affects the next) | Slow to catch up — pipelines run one time range at a time |
| Parallel | Run the pipeline across multiple time ranges simultaneously | When time ranges are independent of each other (e.g., fact tables using insert overwrite) | Can tax the infrastructure of your data processor |
| Bulk | Run the pipeline once with start and end time set to the full backfill range | When logic and infrastructure can handle the full range in one go | May not be suitable for pipelines with run-order dependencies |

## Wait to process the fact data until you are certain most of it has arrived

* Fact data generated outside of your company's servers (e.g. User browser, 3-rd party data vendor) usually have some delay in reaching our data infrastructure.
* Consider that you are capturing user activity on your website.
* The user's wifi may have connectivity issues, the user may close their laptop, etc that causes the data to not be sent as soon as it is generated.
* The difference between when an event was created and when it reaches our data infrastructure is the late-ness of it

![Late Arriving Events](images/late_arriving_events.png)

* Due to this delay its essential to wait for *a while* before we start a pipeline to process a time ranges, data.

#### Example

* If most of the fact data arrives within 2 hours of generation, we can be reasonably sure the pipeline can start 2 hours after the `end_time` of the time range we need to process the data for.

* We can use percentile to establish a reasonable wait time for our pipeline before starting to process the input data.

```sql
SELECT
    DATE_TRUNC('hour', event_ts)                            AS event_hour,
    AVG(TIMESTAMPDIFF(SECOND, event_ts, loaded_at))         AS avg_late_arrival_seconds,
    MAX(TIMESTAMPDIFF(SECOND, event_ts, loaded_at))         AS max_late_arrival_seconds,
    PERCENTILE(TIMESTAMPDIFF(SECOND, event_ts, loaded_at), 0.95) AS p95_late_arrival_seconds
FROM clickstream
GROUP BY 1
ORDER BY 1
```

* The p95 is useful here since late arrival distributions tend to be skewed — a small number of very late events can pull the average up, so percentiles give a clearer picture of typical behaviour.

#### Exercise [5 min]

If you identify that 98% of your data arrives within 2 hours, for a daily pipeline run, when would you schedule the pipeline to run for each day?



* Analysing historical late arrival distributions (e.g. p95 of loaded_at - event_ts) tells us how long to wait to capture n% of data. This delay is then used to set a `watermark`, a threshold that defines how late an event can arrive before it is excluded.

* The decision to use X hours should be based on analyzing arrival times in your data. And how much data are we willing to lose? 

* In most cases, losing a few data points may be ok

#### Exercise [10 min]

Given the sample data (created as CTE below), what would the schedule be for a pipeline to get this data.

In [ ]:
spark.sql("""
WITH clickstream AS (
    SELECT 'evt-001' AS event_id, 'cust-001' AS customer_id, 'page_view'  AS event_type, CAST('2026-02-24 10:00:00' AS TIMESTAMP) AS event_ts, CAST('2026-02-24 10:01:30' AS TIMESTAMP) AS loaded_at
    UNION ALL
    SELECT 'evt-002', 'cust-002', 'click',     CAST('2026-02-24 10:02:00' AS TIMESTAMP), CAST('2026-02-24 10:03:45' AS TIMESTAMP)
    UNION ALL
    SELECT 'evt-003', 'cust-001', 'add_to_cart', CAST('2026-02-24 10:05:00' AS TIMESTAMP), CAST('2026-02-24 10:06:10' AS TIMESTAMP)
    UNION ALL
    SELECT 'evt-004', 'cust-003', 'page_view',  CAST('2026-02-24 10:08:00' AS TIMESTAMP), CAST('2026-02-24 10:45:00' AS TIMESTAMP)  -- outlier, 37 min late
    UNION ALL
    SELECT 'evt-005', 'cust-002', 'purchase',   CAST('2026-02-24 10:10:00' AS TIMESTAMP), CAST('2026-02-24 10:12:20' AS TIMESTAMP)
)
SELECT 1 -- your code here
""").toPandas()

In [ ]:
spark.sql("""
WITH clickstream AS (
    SELECT 'evt-001' AS event_id, 'cust-001' AS customer_id, 'page_view'  AS event_type, CAST('2026-02-24 10:00:00' AS TIMESTAMP) AS event_ts, CAST('2026-02-24 10:01:30' AS TIMESTAMP) AS loaded_at
    UNION ALL
    SELECT 'evt-002', 'cust-002', 'click',     CAST('2026-02-24 10:02:00' AS TIMESTAMP), CAST('2026-02-24 10:03:45' AS TIMESTAMP)
    UNION ALL
    SELECT 'evt-003', 'cust-001', 'add_to_cart', CAST('2026-02-24 10:05:00' AS TIMESTAMP), CAST('2026-02-24 10:06:10' AS TIMESTAMP)
    UNION ALL
    SELECT 'evt-004', 'cust-003', 'page_view',  CAST('2026-02-24 10:08:00' AS TIMESTAMP), CAST('2026-02-24 10:45:00' AS TIMESTAMP)  -- outlier, 37 min late
    UNION ALL
    SELECT 'evt-005', 'cust-002', 'purchase',   CAST('2026-02-24 10:10:00' AS TIMESTAMP), CAST('2026-02-24 10:12:20' AS TIMESTAMP)
)
SELECT
    DATE_TRUNC('hour', event_ts)                            AS event_hour,
    AVG(TIMESTAMPDIFF(SECOND, event_ts, loaded_at))         AS avg_late_arrival_seconds,
    MAX(TIMESTAMPDIFF(SECOND, event_ts, loaded_at))         AS max_late_arrival_seconds,
    PERCENTILE(TIMESTAMPDIFF(SECOND, event_ts, loaded_at), 0.95) AS p95_late_arrival_seconds
FROM clickstream
GROUP BY 1
ORDER BY 1
""").toPandas()

* We saw how we use the bronze and silver layers to model our data. This introduces a new problem.

#### Exercise [5 min]

* Assume daily runs
* Assume we decide to delay the session data pull by 2 hours. And the session pipeline takes 30m to run.
* How long will the data for `fct_session` be delayed by? 

**solution:**

* `bronze.session` starts at 2:30 AM and completes at 3 AM
* fct_session should start at 3:30 AM
* This manual scheduling is fragile. We will see how to handle this in a later section.
* In practice, unless you are operating on a very large scale (think at least a few TBs a day) with unreliable clients (mostly JS that generates async events in browsers or external systems), your systems will not face a significant late-arrival data issue.

## Data pipeline scripts should be re-runnable without creating duplicate or partial data (aka idempotent)

* **Idempotency**: If you run the code multiple times with the same input, the output should be the same. When storing the output in an external data store, it should not be duplicated or partial. Mathematically defined as `f(f(x)) = f(x)`
* To create an idempotent pipeline, there are a few criteria to satisfy.
  * **Atomicity**: A pipeline should only create one table.
  * **No side effects**: A pipeline should not affect any external data (variable or other) besides its output. (with the exception of logs)
* Within a pipeline, it is helpful to have individual functions corresponding to Extract, Transform, & Load, and they can be retried independently.
* Creating data transformations as functions makes the code:
  1. Easy to maintain and debug.
  2. Easy to test.
  3. Represent the reality of transforming data in a series of steps (represented as functions).
  4. Simpler to run in parallel (if your transformations are independent of historical data) with different inputs.

#### Example

* Let’s go over the `dim_customer_snapshot` pipeline. No matter how many times we run it with the same input (data source and time range), the output will be the same. We will not get duplicate rows or partial rows.

In [ ]:
from pyspark.sql import DataFrame, SparkSession

JDBC_URL = "postgre_connection_url"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}
TABLE_NAME = "local.warehouse.dim_customer_snapshot"


def extract(spark: SparkSession) -> dict[str, DataFrame]:
    customer_df = spark.read.jdbc(
        url=url, table="public.customer", properties=properties
    )
    customer_address_df = spark.read.jdbc(
        url=url, table="public.customer_address", properties=properties
    )
    return {"customer_df": customer_df, "customer_address_df": customer_address_df}


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    input_dfs["customer_df"].createOrReplaceTempView("customer")
    input_dfs["customer_address_df"].createOrReplaceTempView("customer_address")

    return spark.sql("""
        SELECT
            c.customer_id,
            c.email,
            c.full_name,
            c.phone,
            c.status,
            c.created_at,
            c.updated_at,
            COLLECT_LIST(
                STRUCT(
                    ca.is_default,
                    CONCAT(ca.line1, ', ', ca.city, ', ', ca.state, ', ', ca.country) AS address
                )
            ) AS addresses
        FROM customer c
        LEFT JOIN customer_address ca USING (customer_id)
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    """)


def load(output_df: DataFrame) -> None:
    output_df.writeTo(TABLE_NAME).createOrReplace()


def run(spark: SparkSession) -> None:
    load(transform(extract(spark)))

In [ ]:
run(spark)

In [ ]:
spark.table("local.warehouse.dim_customer_snapshot").count()

In [ ]:
run(spark)

In [ ]:
spark.table("local.warehouse.dim_customer_snapshot").count()

#### Exercise [5 min]

Is the `fct_order_lines_incremental` pipeline idempotent? Why, why not?

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

TABLE_NAME = "local.warehouse.fct_order_lines_incremental"

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}


def extract(
    spark: SparkSession,
    start_time: str,
    end_time: str,
) -> dict[str, DataFrame]:
    order_line_df = spark.read.jdbc(
        url=JDBC_URL,
        table="public.order_line",
        properties=JDBC_PROPERTIES,
    )
    return {
        "order_line": order_line_df.filter(
            (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
        )
    }


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    return input_dfs["order_line"].select(
        F.col("order_line_id"),
        F.col("order_id"),
        F.col("variant_id"),
        F.col("quantity"),
        F.col("unit_price"),
        F.col("discount_amt"),
        F.col("line_total"),
        F.col("created_at"),
        F.col("updated_at"),
        (F.col("discount_amt") > 0).alias("is_discounted"),
        (F.col("unit_price") - F.col("discount_amt")).alias("effective_unit_price"),
    )


def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        (
            output_df.writeTo(TABLE_NAME)
            .partitionedBy(F.partitioning.days("created_at"))
            .createOrReplace()
        )
    else:
        output_df.writeTo(TABLE_NAME).overwritePartitions()


def run(spark: SparkSession, start_time: str, end_time: str) -> None:
    load(transform(extract(spark, start_time, end_time)), spark)

* Yes, it is idempotent. If we run the pipeline with the same inputs (data source and time ranges), the output will be overwritten, so it will remain the same.

* Is SCD2 with a `MERGE INTO` idempotent?

* No, since if you re-ran the pipeline with updated input the older rows created by the prior runs will remain in dim_customer, thus making the output incorrect. You will need to manually clean up the SCD2 data before re-running SCD2 pipelines.

## Self-healing pipelines make maintenance easy.

* While an idempotent data pipeline may sound like the gold standard, it can be challenging to implement and maintain with changing business requirements and non-replayable sources.
* A more straightforward design pattern is self-healing pipelines.
* The idea behind a self-healing pipeline is that the next pipeline run will “catch up” all the unprocessed data when an error occurs during a run.

![Catchup Pipeline](images/catchup_pattern.png)
* Errors may arise from flaky input data, infrastructure failures, flaky tests, etc.

#### Exercise [5 min]

* How will your pipeline know the time range to process?


**Solution**

* You can store the last successful run’s time interval (end_time) and access that as a start time for the next run's start time



**Pros of self-healing**

1. Simpler to maintain.
2. Since it self-heals, it reduces alert fatigue.
3. Ideal for pipelines where upstream data sources or data infrastructure intermittently fail.
4. The catch-up pipeline can be designed to run failed runs as idempotent runs, thus providing the best of both worlds.

**Cons**

1. Because the pipeline is self-healing, engineers may assume code bugs will resolve themselves, causing some bugs to go unnoticed for a few runs.
2. If you have a failed run, you will need logic to prevent the catch-up run from creating duplicates. You will also need to ensure that partial records are not made available to end users. Use overwrite partitions or overwrite to overcome this issue.
3. Pipeline run times may be inconsistent due to variability in input data volume.
4. Maintaining accurate pipeline metadata is essential to determining when to rerun data pipelines. Most orchestration tools provide mechanisms for handling this metadata.

* We will cover implementation in the scheduler section.

## Recap

In this section we covered pipeline design. We saw 

1. How Python serves as the control layer and the data processing engine does the execution
2. How SCD2 can be implemented with MERGE INTO
3. How to design pipelines for backfills
4. Handling late arriving events
5. Idempotent and self healing pipeline designs

In the next section we will use these principles to create data flows using the medallion architecture